# ECHO-AI — Chest X-ray Inference

This notebook loads a trained DenseNet121 checkpoint and predicts scores for seven chest X-ray findings.

## Required files
- `echo_densenet121.pt` — trained model checkpoint
- One chest X-ray image (`.png`, `.jpg`, or `.jpeg`)

**Important:** This notebook contains inference code, not the trained model weights. If you don't have the checkpoint, predictions from your previously trained model cannot be reproduced until the checkpoint is recovered or the model is retrained.

This is an experimental student project, not a medical device. Outputs are not diagnoses and must not be used for clinical decisions.


In [ ]:
# Install dependencies
%pip install -q torch torchvision pillow


In [ ]:
# Upload the trained checkpoint and one X-ray image
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


In [ ]:
import os
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

DEFAULT_LABELS = [
    "Pneumonia",
    "Cardiomegaly",
    "Effusion",
    "Edema",
    "Pneumothorax",
    "Atelectasis",
    "Consolidation",
]
DEFAULT_THRESHOLDS = [0.50, 0.44, 0.54, 0.34, 0.50, 0.60, 0.64]
DEFAULT_IMAGE_SIZE = 224

# Change these if your uploaded filenames differ.
CHECKPOINT_PATH = "echo_densenet121.pt"
IMAGE_PATH = None  # Set to your image filename, e.g. "chest_xray.png"

print("GPU available:", torch.cuda.is_available())
print("Checkpoint found:", os.path.isfile(CHECKPOINT_PATH))


In [ ]:
def build_model(num_labels=7):
    # Architecture used by ECHO-AI. No pretrained download is needed.
    model = models.densenet121(weights=None)
    model.classifier = nn.Linear(model.classifier.in_features, num_labels)
    return model


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ("state_dict", "model_state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, dict):
                return value
        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint
    raise ValueError(
        "Could not find weights in checkpoint. Expected 'state_dict' "
        "or 'model_state_dict' in the saved checkpoint."
    )


def load_model(checkpoint_path, device):
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}. Upload echo_densenet121.pt first."
        )

    try:
        checkpoint = torch.load(
            checkpoint_path, map_location=device, weights_only=False
        )
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location=device)

    labels = checkpoint.get("labels", DEFAULT_LABELS) if isinstance(checkpoint, dict) else DEFAULT_LABELS
    thresholds = checkpoint.get("thresholds", DEFAULT_THRESHOLDS) if isinstance(checkpoint, dict) else DEFAULT_THRESHOLDS
    image_size = checkpoint.get("img_size", DEFAULT_IMAGE_SIZE) if isinstance(checkpoint, dict) else DEFAULT_IMAGE_SIZE

    labels = list(labels)
    thresholds = [float(value) for value in thresholds]

    model = build_model(len(labels))
    state_dict = extract_state_dict(checkpoint)
    if state_dict and all(key.startswith("module.") for key in state_dict):
        state_dict = {key[len("module."):]: value for key, value in state_dict.items()}

    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model, labels, thresholds, int(image_size)


def predict(image_path, checkpoint_path=CHECKPOINT_PATH):
    if not image_path or not os.path.isfile(image_path):
        raise FileNotFoundError(
            f"X-ray image not found: {image_path}. Upload an image and set IMAGE_PATH."
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, labels, thresholds, image_size = load_model(checkpoint_path, device)

    preprocess = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    image = Image.open(image_path).convert("RGB")
    tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        probabilities = torch.sigmoid(model(tensor))[0].cpu().tolist()

    print(f"Device: {device}")
    print(f"Image: {image_path}\n")
    print(f"{'Condition':<20} {'Probability':>12} {'Threshold':>10}  Result")
    print("-" * 60)
    for label, probability, threshold in zip(labels, probabilities, thresholds):
        result = "Above threshold" if probability >= threshold else "Below threshold"
        print(f"{label:<20} {probability:>11.3f} {threshold:>10.2f}  {result}")

    print("\nWARNING: Experimental output only; not a medical diagnosis.")
    return dict(zip(labels, probabilities))


In [ ]:
# Set IMAGE_PATH to the exact uploaded image filename, then run this cell.
# Example:
# IMAGE_PATH = "chest_xray.png"

if IMAGE_PATH is None:
    print("Set IMAGE_PATH to the filename of your uploaded X-ray image, then rerun this cell.")
else:
    predictions = predict(IMAGE_PATH, CHECKPOINT_PATH)
